# 06 · Workforce observation contract
## One bounded feature-engineering milestone

**Hypothesis:** worker-to-task access, expiry, and the availability of substitute workers need a representation that does not reject or silently truncate legal workforces.

This notebook implements and audits **124 aggregate candidate descriptors**, plus variable-length worker and task tables. It does **not** change a frozen policy, select features, fit a model, or claim a stronger Kaggle score.

**Scope:** 46 fast local tests → pinned-source/data preflight → four mechanics transitions with both seats → the 161 saved final-day observations from seven accepted episodes → five interactive Plotly views → durable receipts.

**Hard limit:** the execution worker is stopped after 180 seconds. Successful episode checkpoints survive failure. No new full games, paid jobs, cloud-resource writes, package installations, or Git operations are called.

Use **Kaggriculture Manual (verified source)**. Run this notebook only, not the historical notebook executor.

In [1]:
from pathlib import Path
import json, sys, subprocess, datetime
from IPython.display import display, Markdown

BASE = Path.cwd().resolve()
if not (BASE / "run_milestone.py").is_file():
    BASE = Path.home() / "kaggriculture_workforce_milestone"
if not (BASE / "run_milestone.py").is_file():
    raise FileNotFoundError("Open this notebook from the extracted kaggriculture_workforce_milestone folder.")
sys.path.insert(0, str(BASE))
STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()
print("WORKFORCE AUDIT — no reference-mode fallback")
print("Python:", sys.executable)
print("Package:", BASE)
print("Started UTC:", STARTED)


WORKFORCE AUDIT — no reference-mode fallback
Python: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python
Package: /home/sagemaker-user/kaggriculture_workforce_milestone
Started UTC: 2026-09-11T22:09:33.876931+00:00


## 1 · What notebook 05 already established

Do not repeat workspace preparation, clone again, erase old worktrees, or download raw files again. The runtime and the seven restored episode objects are reused. The short preflight below checks that their bytes still match.

The two files listed inside your registered Kaggle ZIP are `AGENTS.md` and `README.md`. This audit uses generated research episodes, not a guessed tabular training file. The archive contents/provenance are not independently re-certified here.

In [2]:
import pandas as pd
resume = Path.home() / "kaggriculture_manual_resume"
previous = json.loads((resume / "state/readiness.json").read_text())
display(pd.DataFrame([{
    "Previous readiness": previous["status"],
    "Source": previous["source_commit"],
    "Restored objects": previous["verified_private_objects"],
    "Prior games run": previous["games_run"],
    "Official metric measured": previous["official_metric_measured"],
}]))


,Previous readiness,Source,Restored objects,Prior games run,Official metric measured
0,READY_FOR_ZERO_GAME_FEATURE_EVIDENCE_REVIEW,7194116dfc92a8663139611233b5a221dad431a4,10,0,False


## 2 · Execute the bounded audit

This cell launches a separate worker using this kernel's interpreter. It prints tests, stage receipts, episode counters and 10-second heartbeats. It does not invoke the historical policies. A timeout preserves valid checkpoints and raises an error rather than silently continuing.

Four mechanics transitions reproduce 0-, 4-, and 12-hand cases with both hiring seats and both observer views. **Twelve is not claimed as a maximum.** Separate synthetic stress tests reach 100 hands but are not proof those states are reachable in a game.

**Stop on any exception.** Save the notebook and bundle the outputs for diagnosis; do not retry an unchanged failure repeatedly.

In [3]:
subprocess.run(
    [sys.executable, str(BASE / "run_milestone.py"), "run", "--seconds", "180"],
    cwd=BASE, check=True, timeout=195,
)
from run_milestone import verify_outputs
report = verify_outputs()
assert report["status"] == "WORKFORCE_COVERAGE_PASSED"
assert report["full_games_run"] == 0
print("WORKFORCE_COVERAGE_PASSED")
print("Full-callback acceptance:", report["full_callback_acceptance"])
print("Official metric improvement measured:", report["official_metric_effect_measured"])


{"hard_limit_seconds": 180, "stage": "RUN_STARTED", "utc": "2026-09-11T22:09:34.719080+00:00"}
..............................................
----------------------------------------------------------------------
Ran 46 tests in 0.045s

OK
{"stage": "PREFLIGHT_PASSED", "utc": "2026-09-11T22:09:36.225170+00:00", "verified_objects": 10}
{"hired_hands": 0, "hiring_player": 0, "stage": "MECHANICS", "utc": "2026-09-11T22:09:36.278992+00:00", "views_completed": 2}
{"hired_hands": 4, "hiring_player": 0, "stage": "MECHANICS", "utc": "2026-09-11T22:09:36.286129+00:00", "views_completed": 4}
{"hired_hands": 12, "hiring_player": 0, "stage": "MECHANICS", "utc": "2026-09-11T22:09:36.293774+00:00", "views_completed": 6}
{"hired_hands": 0, "hiring_player": 1, "stage": "MECHANICS", "utc": "2026-09-11T22:09:36.330433+00:00", "views_completed": 8}
{"hired_hands": 4, "hiring_player": 1, "stage": "MECHANICS", "utc": "2026-09-11T22:09:36.337616+00:00", "views_completed": 10}
{"hired_hands": 12, "hiring_pla

## 3 · Inspect representation coverage

The old observation validator is intentionally unchanged. Passing the new extractor does not mean the entire agent now supports larger workforces.

Worker rows retain the actual action index: farmer = 0, then every observed hired hand. Aggregate features do not depend on the order of the hands. Opponent inventory is **unknown**, not zero. The extractor takes only a single current observation and registered public mechanics constants; episode rewards and seeds are not feature inputs.

In [4]:
from visualize import build_figures, export_dashboard
OUT = BASE / "outputs"
figures = build_figures(OUT)
figures[0][1].show()
figures[1][1].show()
display(pd.DataFrame([{
    "Aggregate candidates": report["aggregate_candidate_features"],
    "Saved observations": report["saved_observations"],
    "Variable-length worker rows": report["worker_rows"],
    "Task rows": report["task_rows"],
    "Mechanics views": report["mechanics_views"],
    "Unit tests": report["tests"]["tests_run"],
}]))


,Aggregate candidates,Saved observations,Variable-length worker rows,Task rows,Mechanics views,Unit tests
0,124,161,1204,6885,12,46


## 4 · Distinguish activation, utility and runtime

A nonzero or varying column shows activation only. No correlation with an outcome is used to select or weight these features. The 161 observations come from only seven prior development episodes; individual turns are not independent performance samples.

The timing figure measures the **new extractor alone**, not the complete policy callback and not the earlier failed 500 ms gate. Do not use it to clear the full-callback acceptance requirement.

In [5]:
figures[2][1].show()
figures[3][1].show()
registry = pd.read_csv(OUT / "feature_registry.csv")
display(registry[["feature", "distinct_values", "episodes_with_variation", "status"]])


,feature,distinct_values,episodes_with_variation,status
0,time.remaining_decisions,23,7,candidate_not_ablated
1,time.remaining_actions_today,23,7,candidate_not_ablated
2,own.worker_count,4,7,candidate_not_ablated
3,own.hired_hands,4,7,candidate_not_ablated
4,own.worker_action_budget_today,22,7,candidate_not_ablated
...,...,...,...,...
119,own_inventory.loaded_workers,5,7,candidate_not_ablated
120,own_inventory.max_worker_load,18,7,candidate_not_ablated
121,own_inventory.deposit_overflow_if_all_arrive,1,0,candidate_not_ablated
122,own_inventory.wheat_in_shed,4,7,candidate_not_ablated


## 5 · Inspect worker-to-task geometry

Movement distances and earliest task-action counts are per-task bounds. Eligibility ignores competing tasks, market response and required feeding supplies. Counts are not joint feasible schedules. The manual harvest-to-cash bound includes harvest, travel to shed access and DROP; it excludes overnight automatic deposit.

The state diagram uses one saved observation chosen only for visible task density, not its game outcome. Hover to inspect worker indices and task identifiers.

In [6]:
figures[4][1].show()
example = json.loads((OUT / "example_state.json").read_text())
worker_table = pd.DataFrame(example["workers"])
display(worker_table)


,available_actions_today,care_alternative_gap,care_best_actions,care_independent_reachable_today,care_present,care_second_actions,care_second_present,carried_units,feed_alternative_gap,feed_best_actions,...,water_second_present,weed_alternative_gap,weed_best_actions,weed_independent_reachable_today,weed_present,weed_second_actions,weed_second_present,worker_index,x,y
0,23,1,1,6,1,2,1,0.0,1,1,...,1,0,5,2,1,5,1,0,4,4
1,23,1,1,6,1,2,1,NaN,1,1,...,1,0,5,6,1,5,1,0,4,4


## 6 · Save evidence and stop here

**Continue gate:** all tests, pinned mechanics fixtures, input hashes and all 161 replay states must pass. This is permission to work on action-path integration and full-callback acceptance—not permission to launch a large study or declare feature engineering complete.

**Next experimental question, after integration:** does an equal-staffing control versus one worker-opportunity intervention produce nonzero legal-action divergence and stable paired game outcomes? That experiment is not run by this notebook.

After this final cell, use **File → Save Notebook**, then run the short terminal bundle command in `START_HERE.md`. Bundling after saving includes this executed notebook.

In [7]:
dashboard = export_dashboard(OUT, figures)
execution = {
    "mode": "AWS_SAVED_OBSERVATION_AUDIT",
    "started_utc": STARTED,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "status": report["status"],
    "report_sha256": __import__("hashlib").sha256((OUT / "report.json").read_bytes()).hexdigest(),
    "plot_count": len(figures),
    "full_games_run": 0,
    "models_fit": 0,
    "official_metric_effect_measured": False,
    "full_callback_acceptance": "NOT_RUN",
    "github_updated": False,
    "feature_completion_gate": "OPEN",
}
from run_milestone import write
write(OUT / "notebook_execution.json", execution)
display(execution)
print("Saved dashboard:", dashboard)
print("SAVE THIS NOTEBOOK, then run the bundle command in the instructions.")


{'mode': 'AWS_SAVED_OBSERVATION_AUDIT',
 'started_utc': '2026-09-11T22:09:33.876931+00:00',
 'finished_utc': '2026-09-11T22:09:42.625292+00:00',
 'status': 'WORKFORCE_COVERAGE_PASSED',
 'report_sha256': '5d432186a1e317362dee7a8164d4744a5d81155dd35a0819b200415a0372e80a',
 'plot_count': 5,
 'full_games_run': 0,
 'models_fit': 0,
 'official_metric_effect_measured': False,
 'full_callback_acceptance': 'NOT_RUN',
 'github_updated': False,
 'feature_completion_gate': 'OPEN'}

Saved dashboard: /home/sagemaker-user/kaggriculture_workforce_milestone/outputs/workforce_dashboard.html
SAVE THIS NOTEBOOK, then run the bundle command in the instructions.
